# Phase 6 — Experiment design: a retention offer for KKBox

> ⚠️ **DESIGN ONLY.** KKBox never ran this experiment. No outcome below is real. The only real inputs are the **baseline churn rate** and **monthly eligible volume**, read from the data. Everything else is an assumption, labelled as such.

The full written design is in `docs/experiment_design.md`; this notebook produces its numbers.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from streampulse.db import query, t, save_result, banner, RESULTS_DIR, IS_FIXTURE

banner()
FIG = RESULTS_DIR / "figures"
FIG.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})
from math import ceil
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

## 6.1 Parameters (edit these, then re-run)
Target segment is chosen from Phases 3–5: the group with high churn **and** enough revenue at risk. Default: monthly plans with auto-renew **off** (they must actively re-buy every month).

In [ ]:
SEGMENT_SQL = "not last_is_auto_renew and last_plan_length_bucket = '02. ~1 month'"
SEGMENT_NAME = "Monthly plan, auto-renew off"
ALPHA = 0.05            # false-positive rate we accept (two-sided)
POWER = 0.80            # chance of detecting a real effect of size MDE
SHARE_ENROLLED = 1.0    # share of eligible members put into the test (both arms)
MDE_ABS_GRID = [0.01, 0.02, 0.03, 0.05]   # absolute reductions in churn rate, in proportion units
# ASSUMPTION: the offer = OFFER_DISCOUNT_SHARE off one month's price
OFFER_DISCOUNT_SHARE = 0.5

## 6.2 Baseline from real data

In [ ]:
base = query(f"""
    select count(*) as eligible_per_month,
           avg(case when is_churn then 1.0 else 0.0 end) as baseline_churn,
           avg(last_monthly_price_ntd) as avg_monthly_price_ntd
    from {t('mart_churn_features')}
    where has_history_before_cutoff and {SEGMENT_SQL}""").iloc[0]
p0 = float(base.baseline_churn)
eligible = int(base.eligible_per_month)
price = float(base.avg_monthly_price_ntd)
print(f"{SEGMENT_NAME}: {eligible:,} memberships expiring per month (Mar 2017), baseline churn {p0:.2%}, avg price {price:,.0f} NTD")

## 6.3 Sample size and duration
Two-proportion test, 50/50 split, unit = member. `n` is per arm.

In [ ]:
rows = []
for mde in MDE_ABS_GRID:
    if mde >= p0:
        continue
    h = proportion_effectsize(p0, p0 - mde)                 # Cohen's h
    n = ceil(NormalIndPower().solve_power(effect_size=h, alpha=ALPHA, power=POWER, ratio=1.0, alternative="two-sided"))
    months = 2 * n / (eligible * SHARE_ENROLLED)
    rows.append({"mde_abs_pp": mde * 100, "mde_relative": mde / p0, "treated_churn": p0 - mde,
                 "cohens_h": h, "n_per_arm": n, "total_n": 2 * n, "months_of_expiries_needed": months})
plan = pd.DataFrame(rows)
plan

## 6.4 Is it worth it? Break-even effect (ASSUMPTIONS)
The offer costs money on every treated member, saved or not. It pays back only if enough members are saved. Value of a saved member = price × expected remaining months (1 / churn) — the same constant-churn assumption as Phase 4.

In [ ]:
offer_cost_per_treated = OFFER_DISCOUNT_SHARE * price
value_per_saved = price * (1 / p0)          # ASSUMPTION: constant churn
breakeven_abs = offer_cost_per_treated / value_per_saved
print(f"ASSUMPTION offer cost per treated member: {offer_cost_per_treated:,.0f} NTD")
print(f"ASSUMPTION value of one saved member:      {value_per_saved:,.0f} NTD")
print(f"=> the offer must cut churn by at least {breakeven_abs * 100:.2f} points to break even")
print("Design rule: choose an MDE no larger than the break-even effect, or a 'win' could still lose money.")

In [ ]:
res = {
    "DESIGN_ONLY_NO_REAL_EXPERIMENT": True,
    "segment": SEGMENT_NAME, "segment_sql": SEGMENT_SQL,
    "baseline_churn_rate": p0, "eligible_per_month": eligible, "avg_monthly_price_ntd": price,
    "alpha": ALPHA, "power": POWER, "share_enrolled": SHARE_ENROLLED,
    "sample_size_table": plan.to_dict("records"),
    "ASSUMPTION_offer_discount_share": OFFER_DISCOUNT_SHARE,
    "ASSUMPTION_breakeven_abs_reduction": breakeven_abs,
}
save_result("phase06_experiment_design", res)
res